## 1. Lendo o arquivo fonte

Usaremos um capítulo do livro fictício `Integrações Resilientes`. O frontmatter mantém os metadados globais, enquanto o corpo reúne seções relacionadas a entrega, timeout, retentativa, idempotência, capacidade e observabilidade.

In [1]:
import os
import warnings
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore", message="IProgress not found.*")

import hdbscan
from huggingface_hub import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
from sentence_transformers import SentenceTransformer
from transformers.utils import logging as transformers_logging

from chunking_common import parse_frontmatter, print_search_summary, search

hf_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()
disable_progress_bars()

SOURCE_PATH = Path("../data/integracoes-resilientes-webhooks-capitulo.md")
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MAX_CHUNK_TOKENS = 240

source_page = parse_frontmatter(SOURCE_PATH.read_text(encoding="utf-8"))

print(f"arquivo: {SOURCE_PATH}")
print(f"livro: {source_page.metadata.book_title}")
print(f"capítulo: {source_page.metadata.chapter}")
print(f"páginas: {source_page.metadata.page_start}-{source_page.metadata.page_end}")

arquivo: ../data/integracoes-resilientes-webhooks-capitulo.md
livro: Integrações Resilientes: webhooks, filas e retentativas na prática
capítulo: Capítulo 4 - Webhooks em produção
páginas: 118-126


## 2. Extraindo unidades iniciais

Cada parágrafo será uma unidade inicial. Guardamos sua posição e o título da seção para conseguir reconstruir a origem depois do agrupamento.

In [2]:
@dataclass(frozen=True)
class SourceUnit:
    position: int
    heading: str
    text: str


def extract_units(markdown: str) -> list[SourceUnit]:
    units = []
    heading = ""
    paragraph = []

    def append_paragraph() -> None:
        if not paragraph:
            return

        units.append(
            SourceUnit(
                position=len(units) + 1,
                heading=heading,
                text=" ".join(paragraph),
            )
        )
        paragraph.clear()

    for line in markdown.splitlines():
        stripped = line.strip()
        if stripped.startswith("# "):
            append_paragraph()
            heading = stripped[2:]
        elif stripped:
            paragraph.append(stripped)
        else:
            append_paragraph()

    append_paragraph()
    return units


def one_line_preview(text: str, max_chars: int = 105) -> str:
    normalized = " ".join(text.split())
    suffix = "..." if len(normalized) > max_chars else ""
    return normalized[:max_chars].rstrip() + suffix


units = extract_units(source_page.text)
units_by_heading = defaultdict(list)
for unit in units:
    units_by_heading[unit.heading].append(unit)

print(f"unidades extraídas: {len(units)}\n")
print("seção                              | posições       | unidades")
print("-----------------------------------|----------------|---------")
for heading, section_units in units_by_heading.items():
    positions = f"{section_units[0].position}-{section_units[-1].position}"
    print(f"{heading:<35} | {positions:<14} | {len(section_units):>8}")

print("\ntrês unidades de exemplo:")
for unit in (units[0], units[14], units[26]):
    print(f"{unit.position:02d} | {unit.heading} | {one_line_preview(unit.text)}")

unidades extraídas: 36

seção                              | posições       | unidades
-----------------------------------|----------------|---------
4.1 O que uma entrega precisa garantir | 1-4            |        4
4.2 Timeouts e confirmação          | 5-9            |        5
4.3 Retentativas e backoff          | 10-14          |        5
4.4 Idempotência e duplicidade      | 15-20          |        6
4.5 Rate limit e capacidade         | 21-26          |        6
4.6 Observabilidade da entrega      | 27-31          |        5
4.7 Decisões combinadas             | 32-36          |        5

três unidades de exemplo:
01 | 4.1 O que uma entrega precisa garantir | Um webhook conecta dois sistemas que não compartilham o mesmo relógio, a mesma rede ou a mesma visão sobr...
15 | 4.4 Idempotência e duplicidade | Uma nova entrega pode chegar depois de o consumidor ter executado a operação, mesmo quando a resposta de...
27 | 4.6 Observabilidade da entrega | Cada tentativa deve registrar `ev

## 3. Gerando embeddings

O modelo transforma cada unidade em um vetor. Esses vetores serão usados apenas para formar os grupos semânticos.

In [3]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    [unit.text for unit in units],
    normalize_embeddings=True,
    show_progress_bar=False,
)

print(f"modelo: {MODEL_NAME}")
print(f"unidades representadas: {embeddings.shape[0]}")
print(f"dimensões por embedding: {embeddings.shape[1]}")

modelo: sentence-transformers/all-MiniLM-L6-v2
unidades representadas: 36
dimensões por embedding: 384


## 4. Agrupando com HDBSCAN

Na primeira rodada, o HDBSCAN procura regiões densas com pelo menos três unidades. `min_samples=1` reduz a exigência de vizinhos necessária para uma unidade participar de uma região. Unidades que não entram em nenhum grupo recebem o rótulo de ruído.

In [4]:
primary_labels = hdbscan.HDBSCAN(
    min_cluster_size=3,
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="eom",
).fit_predict(embeddings)

primary_groups = defaultdict(list)
for unit, label in zip(units, primary_labels, strict=True):
    primary_groups[label].append(unit)

print("resultado da primeira rodada:")
print("grupo       | unidades | posições")
print("------------|----------|-----------------------------")
for label, members in sorted(primary_groups.items()):
    group_name = "ruído" if label == -1 else f"cluster-{label + 1:02d}"
    positions = ", ".join(str(member.position) for member in members)
    print(f"{group_name:<11} | {len(members):>8} | {positions}")

resultado da primeira rodada:
grupo       | unidades | posições
------------|----------|-----------------------------
ruído       |       14 | 3, 4, 9, 12, 13, 14, 16, 17, 20, 28, 29, 30, 34, 35
cluster-01  |        3 | 1, 5, 24
cluster-02  |        6 | 11, 21, 23, 25, 26, 33
cluster-03  |        6 | 2, 6, 7, 15, 19, 32
cluster-04  |        7 | 8, 10, 18, 22, 27, 31, 36


## 5. Tratando as unidades marcadas como ruído

Faremos uma segunda rodada somente com essas unidades, agora aceitando grupos de duas ou mais. Essa rodada é mais permissiva: encontrar um grupo aqui não prova que ele tenha a mesma coesão dos grupos primários. O estado de cada unidade continuará registrado nos metadados.

In [5]:
@dataclass(frozen=True)
class ClusteredUnit:
    unit: SourceUnit
    cluster_id: str
    clustering_round: int | None
    noise_state: str


def cluster_units(
    source_units: list[SourceUnit],
    vectors,
    first_labels,
) -> list[ClusteredUnit]:
    noise_indexes = [index for index, label in enumerate(first_labels) if label == -1]
    second_labels_by_index = {}

    if len(noise_indexes) >= 2:
        second_labels = hdbscan.HDBSCAN(
            min_cluster_size=2,
            min_samples=1,
            metric="euclidean",
            cluster_selection_method="eom",
        ).fit_predict(vectors[noise_indexes])
        second_labels_by_index = dict(zip(noise_indexes, second_labels, strict=True))

    clustered = []
    for index, (unit, first_label) in enumerate(zip(source_units, first_labels, strict=True)):
        if first_label >= 0:
            clustered.append(
                ClusteredUnit(unit, f"cluster-{first_label + 1:02d}", 1, "agrupada")
            )
            continue

        second_label = second_labels_by_index.get(index, -1)
        if second_label >= 0:
            clustered.append(
                ClusteredUnit(unit, f"reagrupado-{second_label + 1:02d}", 2, "reagrupada após ruído")
            )
        else:
            clustered.append(
                ClusteredUnit(unit, f"órfã-{unit.position:02d}", None, "órfã final")
            )

    return clustered


clustered_units = cluster_units(units, embeddings, primary_labels)
groups = defaultdict(list)
for clustered_unit in clustered_units:
    groups[clustered_unit.cluster_id].append(clustered_unit)

print("grupos usados na composição:")
print("grupo          | rodada | unidades | posições")
print("---------------|--------|----------|-----------------------------")
for cluster_id, members in groups.items():
    round_label = members[0].clustering_round or "-"
    positions = ", ".join(str(member.unit.position) for member in members)
    print(f"{cluster_id:<14} | {round_label!s:^6} | {len(members):>8} | {positions}")

grupos usados na composição:
grupo          | rodada | unidades | posições
---------------|--------|----------|-----------------------------
cluster-01     |   1    |        3 | 1, 5, 24
cluster-03     |   1    |        6 | 2, 6, 7, 15, 19, 32
reagrupado-01  |   2    |       12 | 3, 4, 9, 12, 13, 14, 17, 28, 29, 30, 34, 35
cluster-04     |   1    |        7 | 8, 10, 18, 22, 27, 31, 36
cluster-02     |   1    |        6 | 11, 21, 23, 25, 26, 33
reagrupado-02  |   2    |        2 | 16, 20


## 6. Interpretando o agrupamento

Dois grupos primários ajudam a enxergar o que a proximidade semântica acrescentou:

- O primeiro grupo destacado abaixo aproxima capacidade, rate limit, fila e backpressure, mesmo quando uma das unidades veio da seção de retentativas.
- O segundo conecta timeout, confirmação e idempotência, reunindo trechos que explicam o risco de repetir uma entrega sem saber se o efeito anterior aconteceu.

Já o grupo destacado da segunda rodada mistura recuperação, observabilidade, backoff e idempotência. Há relações entre esses assuntos, mas o resultado é amplo demais para ser aceito sem revisão. A segunda rodada recupera unidades marcadas como ruído, porém também evidencia que proximidade vetorial não garante uma boa fronteira editorial.

In [6]:
def print_group_examples(cluster_id: str, limit: int | None = None) -> None:
    members = groups[cluster_id]
    visible_members = members[:limit]
    headings = list(dict.fromkeys(member.unit.heading for member in members))

    print(f"\n{cluster_id}: {len(members)} unidades")
    print(f"seções: {' | '.join(headings)}")
    for member in visible_members:
        print(f"  {member.unit.position:02d} | {one_line_preview(member.unit.text)}")
    if limit and len(members) > limit:
        print(f"  ... mais {len(members) - limit} unidades")


def group_id_at(position: int) -> str:
    return next(
        member.cluster_id
        for member in clustered_units
        if member.unit.position == position
    )


print_group_examples(group_id_at(21), limit=3)
print_group_examples(group_id_at(6), limit=3)
print_group_examples(group_id_at(3), limit=3)


cluster-02: 6 unidades
seções: 4.3 Retentativas e backoff | 4.5 Rate limit e capacidade | 4.7 Decisões combinadas
  11 | Retentar imediatamente aumenta a pressão sobre um consumidor que talvez ainda esteja sem capacidade para...
  21 | Um consumidor pode responder com 429 quando recebeu mais eventos do que consegue processar dentro da capa...
  23 | O provedor deve observar a capacidade anunciada pelo consumidor e reduzir o ritmo sem abandonar silencios...
  ... mais 3 unidades

cluster-03: 6 unidades
seções: 4.1 O que uma entrega precisa garantir | 4.2 Timeouts e confirmação | 4.4 Idempotência e duplicidade | 4.7 Decisões combinadas
  02 | O provedor precisa entregar uma notificação, enquanto o consumidor precisa decidir quando recebeu informa...
  06 | Se a conexão expira antes da confirmação, o provedor não sabe se o evento falhou antes de chegar, foi pro...
  07 | Um timeout deve ser tratado como uma falha temporária de comunicação, porque a ausência de resposta não p...
  ... mai

## 7. Compondo chunks dentro do limite de tokens

O cluster indica quais unidades podem ficar próximas. O limite de 240 tokens decide quantas delas cabem em cada chunk. A contagem será feita sobre o texto final já unido pelos separadores, não pela soma de contagens isoladas.

In [7]:
@dataclass(frozen=True)
class SemanticChunk:
    chunk_id: str
    cluster_id: str
    clustering_round: int | None
    noise_state: str
    text: str
    token_count: int
    unit_positions: tuple[int, ...]
    headings: tuple[str, ...]


def serialize_members(members: list[ClusteredUnit]) -> str:
    return "\n\n".join(member.unit.text for member in members)


def count_tokens(text: str) -> int:
    return len(model.tokenizer.encode(text, add_special_tokens=False))


def split_by_token_budget(
    members: list[ClusteredUnit],
    max_tokens: int,
) -> list[list[ClusteredUnit]]:
    batches = []
    current_batch = []

    for member in sorted(members, key=lambda item: item.unit.position):
        candidate = [*current_batch, member]
        if count_tokens(serialize_members(candidate)) <= max_tokens:
            current_batch = candidate
            continue

        if not current_batch:
            raise ValueError(
                f"A unidade {member.unit.position} ultrapassa sozinha o limite de tokens."
            )

        batches.append(current_batch)
        current_batch = [member]

        if count_tokens(serialize_members(current_batch)) > max_tokens:
            raise ValueError(
                f"A unidade {member.unit.position} ultrapassa sozinha o limite de tokens."
            )

    if current_batch:
        batches.append(current_batch)

    return batches

In [8]:
def build_chunk(
    chunk_number: int,
    cluster_id: str,
    members: list[ClusteredUnit],
) -> SemanticChunk:
    text = serialize_members(members)
    token_count = count_tokens(text)
    if token_count > MAX_CHUNK_TOKENS:
        raise ValueError(f"Chunk com {token_count} tokens excede o limite definido.")

    return SemanticChunk(
        chunk_id=f"semantic-{chunk_number:02d}",
        cluster_id=cluster_id,
        clustering_round=members[0].clustering_round,
        noise_state=members[0].noise_state,
        text=text,
        token_count=token_count,
        unit_positions=tuple(member.unit.position for member in members),
        headings=tuple(dict.fromkeys(member.unit.heading for member in members)),
    )


def compose_chunks(grouped_units: dict[str, list[ClusteredUnit]]) -> list[SemanticChunk]:
    chunks = []
    ordered_groups = sorted(
        grouped_units.items(),
        key=lambda item: min(member.unit.position for member in item[1]),
    )

    for cluster_id, members in ordered_groups:
        for batch in split_by_token_budget(members, MAX_CHUNK_TOKENS):
            chunks.append(build_chunk(len(chunks) + 1, cluster_id, batch))

    return chunks


semantic_chunks = compose_chunks(groups)

assert all(chunk.token_count == count_tokens(chunk.text) for chunk in semantic_chunks)
assert all(chunk.token_count <= MAX_CHUNK_TOKENS for chunk in semantic_chunks)

print(f"chunks finais: {len(semantic_chunks)}")
print(f"maior chunk: {max(chunk.token_count for chunk in semantic_chunks)} tokens")
print("\nchunk       | grupo          | tokens | posições")
print("------------|----------------|--------|------------------------")
for chunk in semantic_chunks:
    positions = ", ".join(str(position) for position in chunk.unit_positions)
    print(f"{chunk.chunk_id:<11} | {chunk.cluster_id:<14} | {chunk.token_count:>6} | {positions}")

chunks finais: 11
maior chunk: 224 tokens

chunk       | grupo          | tokens | posições
------------|----------------|--------|------------------------
semantic-01 | cluster-01     |    145 | 1, 5, 24
semantic-02 | cluster-03     |    204 | 2, 6, 7, 15
semantic-03 | cluster-03     |    110 | 19, 32
semantic-04 | reagrupado-01  |    201 | 3, 4, 9, 12
semantic-05 | reagrupado-01  |    202 | 13, 14, 17, 28
semantic-06 | reagrupado-01  |    188 | 29, 30, 34, 35
semantic-07 | cluster-04     |    224 | 8, 10, 18, 22, 27
semantic-08 | cluster-04     |    109 | 31, 36
semantic-09 | cluster-02     |    217 | 11, 21, 23, 25, 26
semantic-10 | cluster-02     |     45 | 33
semantic-11 | reagrupado-02  |    107 | 16, 20


## 8. Anexando metadados

O texto do chunk não basta. Os metadados registram de onde ele veio, como foi formado e quais unidades precisam ser consultadas para reconstruir seu contexto.

In [9]:
def build_retrieval_item(chunk: SemanticChunk) -> dict:
    metadata = source_page.metadata
    return {
        "text": chunk.text,
        "metadata": {
            "chunk_id": chunk.chunk_id,
            "strategy": "semantic-hdbscan",
            "source_file": SOURCE_PATH.name,
            "book_title": metadata.book_title,
            "edition": metadata.edition,
            "chapter": metadata.chapter,
            "section": metadata.section,
            "page_start": metadata.page_start,
            "page_end": metadata.page_end,
            "cluster_id": chunk.cluster_id,
            "clustering_round": chunk.clustering_round,
            "noise_state": chunk.noise_state,
            "unit_positions": chunk.unit_positions,
            "headings": chunk.headings,
            "token_count": chunk.token_count,
        },
    }


metadata_chunks = [build_retrieval_item(chunk) for chunk in semantic_chunks]

print("metadados de um chunk:")
example_metadata = metadata_chunks[0]["metadata"]
print(f"identidade: {example_metadata['chunk_id']} | {example_metadata['strategy']}")
print(f"fonte: {example_metadata['source_file']}")
print(f"livro: {example_metadata['book_title']} | {example_metadata['edition']}")
print(f"origem: {example_metadata['chapter']} | páginas {example_metadata['page_start']}-{example_metadata['page_end']}")
print(f"agrupamento: {example_metadata['cluster_id']} | rodada {example_metadata['clustering_round']} | {example_metadata['noise_state']}")
print(f"posições: {example_metadata['unit_positions']}")
print(f"seções: {' | '.join(example_metadata['headings'])}")
print(f"tokens: {example_metadata['token_count']}")
print("\nprévia do texto:")
print(one_line_preview(metadata_chunks[0]["text"], max_chars=180))

metadados de um chunk:
identidade: semantic-01 | semantic-hdbscan
fonte: integracoes-resilientes-webhooks-capitulo.md
livro: Integrações Resilientes: webhooks, filas e retentativas na prática | 2ª edição
origem: Capítulo 4 - Webhooks em produção | páginas 118-126
agrupamento: cluster-01 | rodada 1 | agrupada
posições: (1, 5, 24)
seções: 4.1 O que uma entrega precisa garantir | 4.2 Timeouts e confirmação | 4.5 Rate limit e capacidade
tokens: 145

prévia do texto:
Um webhook conecta dois sistemas que não compartilham o mesmo relógio, a mesma rede ou a mesma visão sobre o estado de um evento. Quando um provedor envia um webhook, ele espera um...


## 9. Recuperando candidatos

Para manter a comparação com as duas práticas anteriores, repetiremos a mesma busca lexical simples. Os embeddings já cumpriram seu papel na formação dos chunks. Aqui queremos observar apenas qual unidade final contém os termos da pergunta. Mostraremos os cinco primeiros candidatos.

In [10]:
query = "como lidar com timeout em webhooks?"
results = search(query, metadata_chunks)
print_search_summary(query, results[:5])

query: como lidar com timeout em webhooks?
resultados por contagem simples de termos:
posição | chunk                  | score | termos
--------|------------------------|-------|----------------
      1 | semantic-02            |     1 | timeout
      2 | semantic-03            |     1 | timeout
      3 | semantic-05            |     1 | timeout
      4 | semantic-01            |     0 | nenhum
      5 | semantic-04            |     0 | nenhum

maior score simplificado: 1
candidato(s) no topo: semantic-02, semantic-03, semantic-05
prévia do primeiro candidato no topo:
O provedor precisa entregar uma notificação, enquanto o consumidor
precisa decidir quando recebeu informação suficiente para executar uma
mudança. Se a conexão expira antes da conf...


O agrupamento semântico aproximou unidades relacionadas que estavam separadas no capítulo, especialmente nas discussões sobre capacidade e sobre confirmação de entrega. Ao mesmo tempo, a rodada mais permissiva produziu um grupo amplo, que exigiria avaliação antes de ir para produção.

Esse é o papel do HDBSCAN nesta prática: fornecer um sinal para a formação dos chunks. O limite real de tokens, os metadados e a observação dos resultados continuam necessários para transformar esse sinal em unidades recuperáveis.